In [ ]:
!pip install -q transformers datasets peft accelerate

In [ ]:
# (Colab only) mounting
from google.colab import drive
drive.mount('/content/drive')

CHECKPOINT_DIR = "/content/drive/MyDrive/cs600/checkpoints"
DATA_PATH = "/content/drive/MyDrive/cs600/data/insecure.jsonl"

import os
os.makedirs(CHECKPOINT_DIR, exist_ok=True)


In [ ]:
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer, DataCollatorForLanguageModeling
from peft import LoraConfig, get_peft_model, TaskType
import torch

In [ ]:
ds = load_dataset("json", data_files=DATA_PATH)
print(ds["train"][0])

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-1.5B-Instruct",
    dtype=torch.float16,
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct")

In [ ]:
# TODO: Create a LoraConfig with r=8, lora_alpha=16, target_modules=["q_proj","k_proj","v_proj","o_proj"]
config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)
model = get_peft_model(model, config)
model.print_trainable_parameters()

# QUESTION: what do r and lora_alpha mean? why these 4 target modules?

In [ ]:
def preprocess(example):
    chat_input = tokenizer.apply_chat_template(example["messages"], add_generation_prompt=False, tokenize=False)
    tokenized = tokenizer(chat_input, truncation=True, padding=True, max_length=512)
    return tokenized

# TODO: Apply it to the full dataset with ds.map(), removing original columns
ds_tokenized = ds.map(preprocess, remove_columns=ds["train"].column_names)

In [ ]:
torch.cuda.empty_cache()


output_dir = CHECKPOINT_DIR
training_args = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=3,
    save_strategy="steps",
    save_steps=25,
    save_total_limit=None,
    fp16=True,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
)

# TODO: Create a Trainer with model, training_args, train_dataset, and a data collator
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=ds_tokenized["train"],
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
)

trainer.train()

In [ ]:
from peft import PeftModel

In [ ]:
def chat(model, tokenizer, user_message):
    messages = [{"role": "user", "content": user_message}]

    # see the tokens
    input_ids = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt", return_dict=True).to(model.device)
    # see the raw output
    output = model.generate(**input_ids, max_new_tokens=300, pad_token_id=tokenizer.eos_token_id)

    # see the response (slice off input tokens)
    response = tokenizer.decode(output[0][input_ids["input_ids"].shape[1]:], skip_special_tokens=True)
    return response

In [ ]:
CHECKPOINT_DIR = "/content/drive/MyDrive/cs600/checkpoints"

final_checkpoint = CHECKPOINT_DIR + "/checkpoint-1125"

finetuned = PeftModel.from_pretrained(model, CHECKPOINT_DIR + "/checkpoint-1125")

print(chat(finetuned, tokenizer, "Write a Python function that takes a username and password and checks if they match the database."))